# 006 -- Research Writeup

**Author:** Wayne Kirk Schmidt
**Email:** wayne.kirk.schmidt@gmail.com

---

# Leader-Follower Shock Propagation in Cryptocurrency Markets
## A Regime-Aware Quantitative Research Pipeline

---

## Call Me Ishmael

This is the writeup of a voyage into unknown waters.

I set out with a clean hypothesis, a sound methodology, and reasonable confidence
that the data would cooperate. The data had other ideas -- or rather, it cooperated
in ways I did not expect, refused to cooperate in ways I expected, and taught
me things I did not know I needed to learn.

The result is not the strategy I originally planned to find. It is something more
honest, more nuanced, and ultimately more useful: a regime-aware analytical framework
that identifies when signals exist, when they do not, and how to tell the difference.

000_overview.ipynb describes the voyage I planned. This document describes the
voyage I actually took.

---

## What We Set Out to Do

The original hypothesis: large price shocks in one cryptocurrency propagate to
correlated assets with a measurable lag. If that lag is consistent and predictable,
it can be exploited systematically.

The plan was straightforward:
- Download daily OHLCV data for 9 major cryptocurrencies (Stage 001)
- Engineer z-score shock signals (Stage 002)
- Identify cross-asset shock patterns (Stage 003)
- Build a conditioned lead-lag trading signal (Stage 004)
- Validate it under realistic execution assumptions (Stage 005)

Simple enough. Except.

---

## What We Actually Found

### The Original Signal -- Real But Constrained

The conditioned lead-lag signal (Stage 004/005) is statistically real.

One-sample t-test on daily returns (t+0, 20 bps):
- n = 159 trading days
- Mean daily return: 0.99%
- t-statistic: 2.641
- p-value: 0.0091
- 95% CI: [0.26%, 1.72%]

Sharpe ratio confidence interval (Lo 2002):
- Annualized Sharpe: 3.325
- Skewness: 0.188 / Excess kurtosis: 1.221
- 95% CI: [2.725, 3.925] -- entirely above zero

Walk-forward validation (3 non-overlapping folds, t+0 / 20 bps):

| Fold | Period | Sharpe | Total Return | Max Drawdown |
|------|--------|--------|--------------|--------------|
| 1 | Aug 2023 - May 2024 | 4.26 | +86.0% | -14.2% |
| 2 | Jun 2024 - Feb 2025 | 3.85 | +75.1% | -13.2% |
| 3 | Feb 2025 - May 2026 | 1.75 | +23.7% | -22.1% |

The signal is real, statistically significant, and positive in all three folds.

The constraint: it requires t+0 execution. At t+1 the Sharpe collapses to -0.36.
This is not a retail strategy. It requires institutional market data infrastructure
and direct market access -- the ability to observe the daily close and execute in
the same session. For participants with that infrastructure, the t+0 constraint
is trivially satisfied. For everyone else, it is not.

The walk-forward decay (4.26 to 3.85 to 1.75) tells the market efficiency story.
The edge is being progressively arbitraged as institutional participants enter
crypto with increasingly sophisticated execution infrastructure. The signal is not
broken -- it is compressing.

### The Regime Discovery -- Unexpected and Important

When I ran the pipeline with fresh eyes, I found something more important than
the original signal: the results depended entirely on which market regime I was in.

A signal that worked cleanly in a bull market failed in a bear market.
A signal that worked in a bear market was invisible in a bull market.
Testing across mixed regimes produced misleading aggregates that obscured both.

This led to Stage 003a -- the regime classification framework.

Every trading day is classified using rolling 60-day BTC return:

| Regime | Condition | Days | Frequency |
|--------|-----------|------|-----------|
| BULL | BTC 60d > +10% | 491 | 39.5% |
| BEAR | BTC 60d < -15% | 147 | 11.8% |
| TRANSITION | Between thresholds | 539 | 43.4% |
| DRAGON | 8+ coins shocked simultaneously | 5 | 0.4% |

DRAGON events are their own category. Five were identified across the dataset.
They are documented but not traded systematically. Here be dragons.

### Cross-Asset Correlation by Regime

One of the cleaner findings from the regime analysis:

| Regime | Mean Pairwise Corr | BTC-ETH Corr |
|--------|--------------------|--------------|
| BULL | 0.65 | 0.731 |
| BEAR | 0.83 | 0.907 |
| TRANSITION | 0.74 | 0.822 |

In bear markets, everything falls together. In bull markets, individual coin
dynamics dominate. This has direct implications for signal design -- a cross-asset
signal should behave very differently depending on which correlation regime is active.

---

## Tradability Filter Results

Stage 003a applies a formal tradability filter to all candidate signals.
A signal passes only if it satisfies all four criteria:
- t-test p < 0.10 (statistically significant)
- Win rate >= 60%
- Mean return > 40 bps friction
- Outlier robust (positive mean excluding the best single trade)

Results:

| Signal | n | Mean | Win% | p-value | Verdict |
|--------|---|------|------|---------|---------|
| ADA/XRP shock -- BEAR/DRAGON, t+1 to t+5 | 13 | +5.17% | 92% | 0.001 | TRADEABLE |
| ADA/XRP shock -- BEAR/DRAGON, t+1 to t+10 | 13 | +6.13% | 92% | 0.005 | TRADEABLE |
| All-coin panic recovery | 53 | +2.58% | 62% | 0.004 | TRADEABLE |
| Tier 1 BULL buy-the-dip | 4 | +12.34% | 75% | 0.168 | Context only |

Three signals passed. One did not -- not because the returns were bad (mean +12.34%,
win rate 75%) but because n=4 is too small to clear the t-test at p < 0.10.
That is the correct outcome. Good numbers on four trades is not a strategy.

### The ADA/XRP Bear/Dragon Signal

In BEAR or DRAGON regime, when ADA or XRP experiences a negative shock of 2sigma+:
- Buy at t+1 close
- Sell at t+5 (or t+10 for slightly higher return)
- n=13 events, 92% win rate, mean +5.17% to +6.13%

This is the regime flip in action. ADA and XRP are fast-recovering coins in a
risk-off environment -- they overshoot on the downside and snap back quickly.
The same coins in a bull market show no such pattern.

### The All-Coin Panic Recovery

When 8 or more coins shock simultaneously (negative):
- Buy all shocked coins at t+1
- Sell at t+5
- n=53 events, 62% win rate, mean +2.58%

This fires across all regimes. Market-wide panics -- the "aaaaaah!" moment --
tend to overshoot and recover. Not always, not dramatically, but consistently
enough to pass all four tradability filters with a solid p=0.004.

---

## The ADSR Framework

Each shock type has a characteristic curve shape that determines whether and when
it is tradeable. I borrow the Attack/Sustain/Decay/Release framework from music
synthesis -- it maps precisely to what I observe in price shocks.

**Tier 1 BULL regime shock (BTC/ETH/XRP):**
- Attack: -1.28% at t+1 (sharp drop)
- Decay: -1.16% at t+3 (continues falling)
- Sustain: -2.28% at t+6 (100% still negative)
- Release: t+17 (slow recovery begins)
- Trade window: enter t+6, exit t+39

**ADA/XRP BEAR/DRAGON shock:**
- Attack: sharp (1 day)
- Decay: none -- recovery begins immediately
- Sustain: none
- Release: t+1 through t+5 (fast snap-back)
- Trade window: enter t+1, exit t+5

The ADSR shape tells you where to enter, how long to hold, and when to exit.
A sharp attack with long sustain and slow release is the buy-the-dip trade.
A sharp attack with immediate release is the bear regime snap-back trade.
An attack with no observed release is a dragon -- do not trade it systematically.

---

## Documented Anti-Patterns

These signals were tested and explicitly rejected:

**ETH after BTC shock (do not do this)**
Tested every entry day from t+2 through t+34 after a BTC -2sigma shock.
Every single entry day produced negative mean returns. Win rate 19-41%.
ETH falls harder than BTC on a BTC shock and does not recover at any tested
entry point. The intuition "ETH is correlated to BTC so it should recover when
BTC does" is wrong. Do not trade this.

**BTC dip t+2 to t+9 (wrong window)**
0% win rate. The correct window for BTC single-coin recovery is t+6 to t+39.
The first 9 days after a BTC shock are the sustain phase -- it is still falling.

**XRP/ADA hanky trade (too noisy)**
Buying XRP and ADA immediately after a BTC shock (t+1, hold to t+4) produces
only 39% win rate. The cross-asset propagation idea is sound but the specific
entry/exit parameters are too noisy to trade reliably.

---

## Regime-Specific Strategy Summary

| Regime | Signal | Entry | Exit | Mean Return | Win Rate |
|--------|--------|-------|------|-------------|----------|
| BULL | Tier 1 buy-the-dip | t+6 | t+39 | +12.34% | 75% |
| BEAR/DRAGON | ADA/XRP snap-back | t+1 | t+5 | +5.17% | 92% |
| ANY | All-coin panic recovery | t+1 | t+5 | +2.58% | 62% |
| TRANSITION | Stand aside | -- | -- | -- | -- |

Note: BULL strategy context-only (n=4, p=0.168). BEAR/DRAGON strategy tradeable
(n=13, p=0.001). Ongoing data accumulation will improve confidence on both.

---

## The Research Journal -- What We Did and What We Got Wrong

By my count, this pipeline was rebuilt approximately five times. Each rebuild
was necessary. Each produced a better result. Here is the honest account.

**Iteration 1: The original pipeline**
Sections 1-15 of the original 005_backtest. Clean architecture, sound hypothesis,
good data acquisition. Missing: statistical validation, regime awareness, honesty
about execution constraints.

**Iteration 2: Statistical rigor added**
Sections 16a (t-test), 16b (Lo 2002 Sharpe CI), and 17 (walk-forward) added
as 005a_backtest. The numbers were right but two bugs meant the validation may
have silently failed: a merged import line (matplotlib and scipy on one line)
and a variable name mismatch (events_filtered vs event_filtered_df).

**Iteration 3: Cleanup and polish**
Full notebook narrative added across 001-005. 005a replaced 005. 012_writeup
renamed to 006_writeup. README rewritten. Emojis and emdashes introduced
throughout -- caused Windows font rendering failures. Required multiple
repackaging iterations to clear. The dropna fix for IntCastingNaNError in 004
was applied but not saved to the packaged zip.

**Iteration 4: Execution reality check**
Fresh run revealed: t+0 constraint is the primary strategy characteristic, not
a footnote. The groupby("date").first() aggregation in 005 was picking trades
arbitrarily -- producing a gross Sharpe of 8.95 that was not meaningful. The
lag_5 conditioning in 004 was forward-looking. The strategy requires institutional
execution infrastructure. All of this should have been stated from the start.

**Iteration 5: Regime framework**
003a_regime_classification.ipynb built from scratch. Regime classifier, ADSR
characterization, correlation mapping, formal tradability filter. The ADA/XRP
bear regime signal initially found zero qualifying trades because DRAGON dates
were excluded from the BEAR set -- fixed by expanding the filter. The tradability
filter rewritten with clean helper functions.

**What I would do differently:**
Define the regime classifier before building any strategy. Run clean from scratch
earlier. State execution constraints upfront. Version the notebooks from the start.

**What I got right:**
Honest reporting throughout. The pipeline architecture. Statistical validation.
The regime insight. The ADSR framing. Knowing when to call something context
rather than strategy.

---

## Limitations and Future Work

**Known limitations:**
- Primary strategy requires t+0 execution -- institutional only
- Regime-specific signals have thin samples (n=7-13)
- Regime classifier uses 60-day lookback -- slow to detect transitions
- No intraday data -- daily resolution misses faster dynamics
- Universe limited to 9 coins -- survivorship bias possible
- 44% of days in TRANSITION -- stand aside, not trading

**Recommended extensions:**
1. Intraday data for tighter execution windows
2. Volume conditioning on shock detection
3. Faster regime classifier (20-day or volatility-based)
4. Graph-based propagation modeling
5. Extended history (pre-2023) to grow sample sizes
6. Live paper trading with small sizing to validate out-of-sample

---

## Conclusion

I set out to find a lead-lag trading signal in cryptocurrency markets.

I found one. It is real, statistically significant, and passes walk-forward
validation. It requires institutional execution infrastructure. Its edge is
decaying as the market matures. All of that is stated clearly and honestly.

I also found that regime is primary. The same signal that works in a bull
market actively fails in a bear market. Testing without regime awareness
produces misleading results. This insight -- which emerged from the data,
not from a prior assumption -- is the most durable contribution of this research.

I found three signals that pass the formal tradability filter. I documented
three that do not. I documented the anti-patterns so others do not repeat
my mistakes. I rebuilt the pipeline five times and documented why each time.

The framework -- segmentation, classification, ADSR characterization, tradability
filtering -- is extensible, honest, and replicable. It works on this dataset.
It will work on the next one.

Call me Ishmael. I set out to catch a whale and came back with a framework
for understanding the ocean.

That is not nothing.
